## Baixar arquivos

In [ ]:
#corpus com documentos
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz
!tar -xvzf collection.tar.gz

In [11]:
#arquivo com questoes
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/qrels.dev.tsv
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/qrels.train.tsv

--2026-05-26 11:43:28--  https://msmarco.z22.web.core.windows.net/msmarcoranking/qrels.dev.tsv
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 57.150.146.100
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|57.150.146.100|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1201626 (1.1M) [text/tab-separated-values]
Saving to: ‘qrels.dev.tsv’

qrels.dev.tsv       100%[===================>]   1.15M   688KB/s    in 1.7s    

2026-05-26 11:43:31 (688 KB/s) - ‘qrels.dev.tsv’ saved [1201626/1201626]

--2026-05-26 11:43:31--  https://msmarco.z22.web.core.windows.net/msmarcoranking/qrels.train.tsv
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 57.150.146.100
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|57.150.146.100|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10589532 (10M) [text/tab-separated-values]
Savin

In [ ]:
#textos das questoes
!wget https://msmarco.z22.web.core.windows.net/msmarcoranking/queries.tar.gz
!tar -xvzf queries.tar.gz

In [3]:
!wget https://zenodo.org/records/15753974/files/__rankgpt-colbert-2000-sampled-100__msmarco-passage-train-judged.run?download=1

zsh:1: no matches found: https://zenodo.org/records/15753974/files/__rankgpt-colbert-2000-sampled-100__msmarco-passage-train-judged.run?download=1


## Carregar Dados

In [1]:
#colocar caminhos
collection_path = "collection.tsv"
qrels_path = "qrels.train.tsv"
hard_negatives_path = "__rankgpt-colbert-2000-sampled-100__msmarco-passage-train-judged.run"

In [2]:
import pandas as pd
#read parquet
df_hard_negatives = pd.read_csv(hard_negatives_path, sep="\t", header=None)
df_hard_negatives.tail()

,0,1,2,3,4,5
199995,1139086,Q0,928581,96,5.0,rank_gpt
199996,1139086,Q0,569931,97,4.0,rank_gpt
199997,1139086,Q0,292317,98,3.0,rank_gpt
199998,1139086,Q0,4382394,99,2.0,rank_gpt
199999,1139086,Q0,3845023,100,1.0,rank_gpt


In [3]:
corpus = pd.read_csv(collection_path, sep="\t", header=None)
corpus.head()

,0,1
0,0,The presence of communication amid scientific ...
1,1,The Manhattan Project and its atomic bomb help...
2,2,Essay on The Manhattan Project - The Manhattan...
3,3,The Manhattan Project was the name for a proje...
4,4,versions of each volume as well as complementa...


In [4]:
qrels = pd.read_csv(qrels_path, sep="\t", header=None)
qrels.head()

,0,1,2,3
0,1185869,0,0,1
1,1185868,0,16,1
2,597651,0,49,1
3,403613,0,60,1
4,1183785,0,389,1


In [5]:
import os

queries_text = pd.read_csv("queries.train.tsv", sep="\t", header=None, names=["qid", "query"])
print(f"Queries carregadas: {len(queries_text)}")
queries_text.head()

Queries carregadas: 808731


,qid,query
0,121352,define extreme
1,634306,what does chattel mean on credit history
2,920825,what was the great leap forward brainly
3,510633,tattoo fixers how much does it cost
4,737889,what is decentralization process.


## Hard Negative dataset construction

In [ ]:
#dataset we want is a list of dicts. The dict has a query field, and a passages field. The passages field is a list of passage(str).
# We will use df_hard_negatives to construct the dataset. The cols are: qid	q0	docno	rank	score	system
# The queries com from the queries dataframe, and the passages come from the corpus dataframe. 
# We will need to iterate over the df_hard_negatives, since each line is only one passage for one query. We will need to group the passages by query.
# We will use the qid to get the query, and the docno to get the passage. We will also use the score to sort the passages. We will only keep the top N passages for each query.

In [6]:
N = 10

# Renomear colunas
df_hard_negatives.columns = ["qid", "q0", "docno", "rank", "score", "system"]
corpus.columns = ["pid", "passage"]
qrels.columns = ["qid", "iteration", "pid", "relevance"]

print("Colunas renomeadas com sucesso.")
print(f"Hard negatives: {len(df_hard_negatives)} linhas, {df_hard_negatives['qid'].nunique()} queries únicas")
print(f"Corpus: {len(corpus)} passagens")
print(f"Qrels: {len(qrels)} linhas")


Colunas renomeadas com sucesso.
Hard negatives: 200000 linhas, 2000 queries únicas
Corpus: 8841823 passagens
Qrels: 532761 linhas


In [7]:
# Construir lookups para acesso rápido por id
corpus_lookup = corpus.set_index("pid")["passage"].to_dict()
query_lookup = queries_text.set_index("qid")["query"].to_dict()

# qrels: positivos (passagens relevantes) agrupados por qid
qrels_positives = qrels.groupby("qid")["pid"].apply(list).to_dict()

# Ordenar por rank (crescente = rank 1 primeiro) e manter top N por query
df_topN = (
    df_hard_negatives
    .sort_values(["qid", "rank"])
    .groupby("qid")
    .head(N)
)

dataset = []
skipped = 0

for qid, group in df_topN.groupby("qid"):
    # Ignorar queries sem texto
    if qid not in query_lookup:
        skipped += 1
        continue

    hard_neg_pids = group["docno"].tolist()
    positive_pids = set(qrels_positives.get(qid, []))

    # Montar lista de passagens: positivas primeiro, depois hard negatives
    # Evitar duplicatas (um pid positivo pode aparecer também no run file)
    all_pids = list(positive_pids) + [pid for pid in hard_neg_pids if pid not in positive_pids]

    passage_texts = []
    is_selected = []

    for pid in all_pids:
        if pid not in corpus_lookup:
            continue
        passage_texts.append(corpus_lookup[pid])
        is_selected.append(1 if pid in positive_pids else 0)

    # answers: textos dos positivos
    answers = [corpus_lookup[pid] for pid in positive_pids if pid in corpus_lookup]

    dataset.append({
        "query_id": qid,
        "query": query_lookup[qid],
        "answers": answers,
        "passages": {
            "is_selected": is_selected,
            "passage_text": passage_texts,
        },
    })

print(f"Dataset construído: {len(dataset)} queries | {skipped} queries sem texto ignoradas")
entry = dataset[0]
print(f"\nExemplo de entrada:")
print(f"  query_id : {entry['query_id']}")
print(f"  query    : {entry['query'][:80]}...")
print(f"  answers  : {len(entry['answers'])} respostas")
print(f"  passages : {len(entry['passages']['passage_text'])} passagens | is_selected={entry['passages']['is_selected']}")


Dataset construído: 2000 queries | 0 queries sem texto ignoradas

Exemplo de entrada:
  query_id : 24
  query    :  both dna and rna are polymers that are made up of ...
  answers  : 1 respostas
  passages : 11 passagens | is_selected=[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [8]:
len(dataset)

2000

In [9]:
dataset[20]

{'query_id': 16155,
 'query': 'amariah meaning',
 'answers': ['Amariah (given name) Amariah is a Hebrew male name meaning God has said or Promised by God. Variations can include Amarissa, Amaris, or Amarit. Amariah is different from Amaria, a Greek female name meaning Moon, Alluring, Pure, Illuminating.. Amariah may refer to: Amariah, biblical characters.'],
 'passages': {'is_selected': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  'passage_text': ['Amariah (given name) Amariah is a Hebrew male name meaning God has said or Promised by God. Variations can include Amarissa, Amaris, or Amarit. Amariah is different from Amaria, a Greek female name meaning Moon, Alluring, Pure, Illuminating.. Amariah may refer to: Amariah, biblical characters.',
   'Amariah The name Amariah is a baby boy name. Meaning Biblical Meaning: The name Amariah is a Biblical baby name. In Biblical the meaning of the name Amariah is: The Lord says; the integrity of the Lord.',
   'Biblical Meaning: The name Amariah is a Biblical

In [10]:
import json

output_path = "hard_negatives_dataset.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

print(f"Dataset salvo em: {output_path}")
print(f"Total de entradas: {len(dataset)}")


Dataset salvo em: hard_negatives_dataset.json
Total de entradas: 2000
